In [ ]:
'''
import os
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

for cnt in range(1, 5):
    url = "https://investors.coca-colacompany.com/filings-reports/annual-filings-10-k?page=" + str(cnt)
    
    #If there is no such folder, the script will create one automatically
    folder_location = r'C:\Users\Aviral Verma\anayltics_vidhya_assignment\RAG systems using LlamaIndex'
    if not os.path.exists(folder_location):os.mkdir(folder_location)
    
    response = requests.get(url)
    soup= BeautifulSoup(response.text, "html.parser")     
    for link in soup.select("a[href$='.pdf']"):
        #Name the pdf files using the last portion of each link which are unique in this case
        filename = os.path.join(folder_location,link['href'].split('/')[-1])
        with open(filename, 'wb') as f:
            f.write(requests.get(urljoin(url,link['href'])).content)
'''

In [1]:
import pandas as pd
import os
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

date_col = []
for cnt in range(1, 5):
    url = "https://investors.coca-colacompany.com/filings-reports/annual-filings-10-k?page=" + str(cnt)
    html_tables = pd.read_html("https://investors.coca-colacompany.com/filings-reports/annual-filings-10-k?page=" + str(cnt))
    df = html_tables[0]
    for date in df['Date']:
        date_col.append(date)

In [2]:
date_col

['03/18/24',
 '02/20/24',
 '02/21/23',
 '02/22/22',
 '02/25/21',
 '02/24/20',
 '02/21/19',
 '02/23/18',
 '02/24/17',
 '02/25/16',
 '02/25/15',
 '02/27/14',
 '02/27/13',
 '02/23/12',
 '02/28/11',
 '02/26/10',
 '02/26/09',
 '02/28/08',
 '02/21/07',
 '02/28/06',
 '03/08/05',
 '03/04/05',
 '03/04/04',
 '02/27/04',
 '03/26/03',
 '03/05/03',
 '03/13/02',
 '03/11/02',
 '03/04/02',
 '03/07/01',
 '03/09/00',
 '03/29/99',
 '03/09/98',
 '03/11/97',
 '03/14/96',
 '03/13/95',
 '03/14/94']

In [3]:
from llama_index.core import VectorStoreIndex, SummaryIndex, SimpleKeywordTableIndex, SimpleDirectoryReader
from llama_index.core.schema import IndexNode
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.openai import OpenAI

In [4]:
import glob, os

In [5]:
# Load all wiki documents
coco_cola_filling = []   
for filename in glob.glob("C:\\Users\\Aviral Verma\\anayltics_vidhya_assignment\\RAG systems using LlamaIndex\\*.pdf"):
    coco_cola_filling.append(SimpleDirectoryReader(input_files=[filename]).load_data()) 

In [6]:
llm = OpenAI(temperature=0, model="gpt-3.5-turbo")

In [7]:
from llama_index.agent.openai import OpenAIAgent

In [8]:
import random
import nest_asyncio
nest_asyncio.apply()

In [14]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import HierarchicalNodeParser, SentenceSplitter
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.storage import StorageContext
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.schema import TextNode
import chromadb
from llama_index.core.retrievers.auto_merging_retriever import AutoMergingRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.indices.vector_store.retrievers import VectorIndexAutoRetriever
from llama_index.core.vector_stores.types import MetadataInfo, VectorStoreInfo
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.core.llama_dataset.generator import RagDatasetGenerator
from llama_index.core.llama_dataset import LabelledRagDataset
from llama_index.llms.openai import OpenAI
from llama_index.core.evaluation import BatchEvalRunner
import nest_asyncio
from llama_index.core.query_engine import SubQuestionQueryEngine

def agent_generator(type):
    cnt = 0
    sample = []
    agents = {}
    query_engine_tools = []
    for document in coco_cola_filling:
        print(cnt)
        try:
            vector_index = ""
            vector_query_engine = ""
            if type == "Sentence Window Retriever":
                embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
                node_parser = SentenceWindowNodeParser.from_defaults(window_size=3,window_metadata_key="window",original_text_metadata_key="original_text")
                nodes = node_parser.get_nodes_from_documents(document)
                sample.append(random.sample(nodes,1))
                vector_index = VectorStoreIndex(nodes, embed_model=embed_model, show_progress=True)
                vector_query_engine = vector_index.as_query_engine(similarity_top_k=2,node_postprocessors=[MetadataReplacementPostProcessor(target_metadata_key="window")])
            elif type == "Auto Merging Retriever":
                node_parser = HierarchicalNodeParser.from_defaults()
                nodes = node_parser.get_nodes_from_documents(document)
                leaf_nodes = get_leaf_nodes(nodes)
                root_nodes = get_root_nodes(nodes)
                sample.append(root_nodes[:2])
                docstore = SimpleDocumentStore()
                docstore.add_documents(nodes)
                storage_context = StorageContext.from_defaults(docstore=docstore)
                vector_index = VectorStoreIndex(leaf_nodes,embed_model = OpenAIEmbedding(model='text-embedding-3-small'),storage_context=storage_context)
                base_retriever = vector_index.as_retriever(similarity_top_k=6)
                retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True)
                vector_query_engine = RetrieverQueryEngine.from_args(retriever)
            elif type == "Auto Retriever":
                chroma_client = chromadb.EphemeralClient()
                chroma_collection = chroma_client.get_or_create_collection("quickstart")
                vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
                storage_context = StorageContext.from_defaults(vector_store=vector_store)
                nodes = []
                node_cnt = 0
                for date in date_col:
                    nodes.append(TextNode(text=(coco_cola_filling[node_cnt][0].text),metadata={"date_filing": date}))
                    node_cnt = node_cnt + 1
                sample.append(random.sample(nodes,1))
                vector_index = VectorStoreIndex(nodes, storage_context=storage_context)
                vector_store_info = VectorStoreInfo(content_info="10K fillings Coco Cola",metadata_info=[MetadataInfo(name="date_filing",type="str",description=("Filling of Investor Return"))])
                vector_query_engine = vector_index.as_query_engine(similarity_top_k=2)
                
            
    
                
            
            # define tools
            query_engine_tools.append(QueryEngineTool(query_engine=vector_query_engine,metadata=ToolMetadata(name="vector_tool",description=("Useful for summarization questions related to"f" {document}")))) 
                
            
            
            # build agent
            function_llm = OpenAI(model="gpt-3.5-turbo")
            agent = OpenAIAgent.from_tools(
                [query_engine_tools[-1]],
                llm=function_llm,
                verbose=True,
            )
            
            agents[date_col[cnt]] = agent
            cnt = cnt + 1
        except:
            pass
    s_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=query_engine_tools)
    agents["99/99/99"] = OpenAIAgent.from_tools([s_engine],llm=function_llm,verbose=True)
    final_sample = []
    for s in sample:
        for node in s:
            final_sample.append(node)
            
    
    return [agents, final_sample]

In [15]:
ret = [ "Auto Retriever","Auto Merging Retriever", "Sentence Window Retriever"]
gpt4 = OpenAI(model='gpt-4o')

In [16]:
def get_eval_results(key, eval_results):
    results = eval_results[key]
    correct = 0
    for result in results:
        if result.passing:
            correct += 1
    score = correct / len(results)
    print(f"{key} Score: {score}")
    return score

In [17]:
from llama_index.core.evaluation import (
    CorrectnessEvaluator,
    RelevancyEvaluator,
    FaithfulnessEvaluator,
)

In [18]:
for method in ret:
    print(method)
    agents, sample = agent_generator(method)
    dataset_generator = RagDatasetGenerator(sample,llm=gpt4,show_progress=True,num_questions_per_chunk=3)    
    eval_dataset = dataset_generator.generate_dataset_from_nodes()
    eval_questions = [example.query for example in eval_dataset.examples]
    eval_answers = [example.reference_answer for example in eval_dataset.examples]
    vector_index = VectorStoreIndex(sample)
    query_engine = vector_index.as_query_engine()
    relevancy_evaluator = RelevancyEvaluator(llm=gpt4)
    faithfulness_evaluator = FaithfulnessEvaluator(llm=gpt4)
    correctness_evaluator = CorrectnessEvaluator(llm=gpt4)
    runner = BatchEvalRunner({"faithfulness": faithfulness_evaluator,"relevancy": relevancy_evaluator,"correctness": correctness_evaluator},workers=8)
    eval_results = await runner.aevaluate_queries(query_engine, queries=eval_questions, reference = eval_answers)
    get_eval_results("faithfulness", eval_results)
    get_eval_results("relevancy", eval_results)    
    get_eval_results("correctness", eval_results)

Auto Retriever
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36


100%|██████████| 3/3 [00:01<00:00,  1.52it/s]


faithfulness Score: 0.6576576576576577
relevancy Score: 0.6846846846846847
correctness Score: 0.5495495495495496
Auto Merging Retriever
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36


100%|██████████| 3/3 [00:02<00:00,  1.08it/s]


faithfulness Score: 0.7522522522522522
relevancy Score: 0.6486486486486487
correctness Score: 0.45495495495495497
Sentence Window Retriever
0


...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1424 [00:00<?, ?it/s]

33
33
33
33


100%|██████████| 3/3 [00:07<00:00,  2.44s/it]


faithfulness Score: 0.8080808080808081
relevancy Score: 0.8181818181818182
correctness Score: 0.797979797979798
